In [15]:
# --- Imports ---

import re
import time
import random
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import current_thread
from seleniumbase import Driver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [24]:
# --- Global Variables ---

# Define the base URL and the specific path for Shopify developers on Clutch.
BASE_URL = "https://clutch.co"
url_param = "developers/shopify"
url = urljoin(BASE_URL, url_param)

# Parallelism settings.
LINK_COLLECTION_WORKERS = 2
HTML_SAVE_WORKERS = 4

# --- SeleniumBase Initialization (UC Mode + Headless) ---
driver = Driver(
    uc=True,
    headless=True,
    incognito=True,
)
driver.get(url)

print("SeleniumBase UC driver started in headless mode.")

# Capture the last page number.
last_page_number = None

SeleniumBase UC driver started in headless mode.


### Functions

In [ ]:
def collect_links_from_page_with_worker(page_number, base_url):
    """Collect profile links from a single page using a dedicated worker driver."""

    worker_driver = Driver(
        uc=True,
        headless=True,
        incognito=True,
    )
    thread_name = current_thread().name

    try:
        print(f"[WORKER START] {thread_name} collecting links from page {page_number}")

        if page_number == 1:
            page_url = base_url
        else:
            separator = "&" if "?" in base_url else "?"
            page_url = f"{base_url}{separator}page={page_number}"

        worker_driver.get(page_url)

        # Wait until company profile cards are visible on the current page.
        WebDriverWait(worker_driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a.provider__title-link.directory_profile[href]")
            )
        )

        page_elements = worker_driver.find_elements(
            By.CSS_SELECTOR, "a.provider__title-link.directory_profile[href]"
        )

        links = []
        for element in page_elements:
            href = element.get_attribute("href")
            if href:
                links.append(href)

        print(f"[WORKER DONE] {thread_name} collected page {page_number}")
        return {
            "page_number": page_number,
            "links": links,
            "error": None,
            "thread_name": thread_name,
        }
    except Exception as exc:
        print(f"[WORKER ERROR] {thread_name} failed collecting page {page_number}: {exc}")
        return {
            "page_number": page_number,
            "links": [],
            "error": str(exc),
            "thread_name": thread_name,
        }
    finally:
        worker_driver.quit()

In [17]:
def capture_number_pages(driver, max_attempts=4):
    """Capture the last pagination number using Selenium only."""

    page_number_pattern = re.compile(r'data-page="(\d+)"')

    for attempt in range(1, max_attempts + 1):
        try:
            # Ensure DOM is available before reading pagination links.
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            # Scroll to trigger lazy-loaded pagination controls.
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.0)

            pagination_links = driver.find_elements(
                By.CSS_SELECTOR,
                'a.sg-pagination-v2-page-number[data-page]'
            )
            data_page_values = []

            for link in pagination_links:
                value = link.get_attribute("data-page")
                if value and value.isdigit():
                    data_page_values.append(int(value))

            # Selenium fallback: read data-page values directly from rendered HTML.
            if not data_page_values:
                html_matches = page_number_pattern.findall(driver.page_source)
                data_page_values = [int(match) for match in html_matches]

            if data_page_values:
                last_page_number = max(data_page_values)
                print(f"Pagination detected on attempt {attempt}. Last page: {last_page_number}")
                return last_page_number

            raise ValueError("No data-page values found in pagination")
        except Exception as exc:
            print(f"Pagination not ready on attempt {attempt}/{max_attempts}: {exc}")
            if attempt == max_attempts:
                raise
            time.sleep(2 ** attempt)

In [25]:
def collect_company_profile_links(url, last_page_number, driver):
    """Collect company profile links from all pages using parallel execution."""

    if last_page_number is None:
        raise ValueError("last_page_number must be defined before collecting company links.")

    company_links = []
    seen_links = set()
    total_pages = last_page_number
    completed = 0

    print(f"Starting link collection with {LINK_COLLECTION_WORKERS} workers across {total_pages} pages...")

    with ThreadPoolExecutor(max_workers=LINK_COLLECTION_WORKERS) as executor:
        future_map = {
            executor.submit(collect_links_from_page_with_worker, page_number, url): page_number
            for page_number in range(1, total_pages + 1)
        }

        for future in as_completed(future_map):
            completed += 1
            result = future.result()
            thread_name = result.get("thread_name", "unknown-thread")
            page_number = result["page_number"]

            if result["error"] is None:
                new_links = 0
                for href in result["links"]:
                    if href and href not in seen_links:
                        seen_links.add(href)
                        company_links.append(href)
                        new_links += 1

                print(
                    f"[PROGRESS] {completed}/{total_pages} pages collected by {thread_name} "
                    f"(page {page_number}, new links: {new_links})"
                )
            else:
                print(
                    f"[PROGRESS] {completed}/{total_pages} pages failed by {thread_name}: "
                    f"page {page_number} | {result['error']}"
                )

    return company_links

In [19]:
def save_page_html_to_bronze(
    page_url,
    driver,
    bronze_dir="../data/bronze",
    max_retries=3,
    initial_delay=1.0,
    backoff_factor=2.0,
):
    """Save page HTML to bronze with automatic exponential backoff retry."""

    import os

    # Extract the profile slug from '/profile/<slug>' in the URL.
    parsed_url = urlparse(page_url)
    path = parsed_url.path.rstrip("/")
    if "/profile/" in path:
        profile_slug = path.split("/profile/")[-1]
    else:
        profile_slug = path.split("/")[-1] or "page"

    # Build a filesystem-safe file name with a high-resolution timestamp.
    safe_slug = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in profile_slug)

    os.makedirs(bronze_dir, exist_ok=True)

    for attempt in range(1, max_retries + 1):
        try:
            # Open the target page and wait until the DOM is available.
            driver.get(page_url)
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )

            timestamp = time.strftime("%Y%m%d_%H%M%S")
            millis = int((time.time() % 1) * 1000)
            file_name = f"{safe_slug}_{timestamp}_{millis:03d}.html"
            file_path = os.path.join(bronze_dir, file_name)

            with open(file_path, "w", encoding="utf-8") as html_file:
                html_file.write(driver.page_source)

            thread_name = current_thread().name
            print(f"HTML for {profile_slug} saved at {time.strftime('%H:%M:%S')} by {thread_name}")
            return file_path
        except Exception as exc:
            if attempt == max_retries:
                print(f"[ERROR] Failed after {max_retries} attempts: {page_url} | {exc}")
                raise

            wait_seconds = initial_delay * (backoff_factor ** (attempt - 1))
            print(
                f"[RETRY] Attempt {attempt}/{max_retries} failed for {page_url}. "
                f"Waiting {wait_seconds:.1f}s..."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(f"Failed to save HTML for {page_url}")

In [20]:
def create_headless_driver():
    """Create a dedicated SeleniumBase UC headless driver."""

    return Driver(
        uc=True,
        headless=True,
        incognito=True,
    )

In [21]:
def save_company_page_with_worker(company_url, bronze_dir="../data/bronze"):
    """Create a worker driver, save one company page, and close the driver."""

    worker_driver = create_headless_driver()
    thread_name = current_thread().name
    try:
        print(f"[WORKER START] {thread_name} handling {company_url}")
        saved_path = save_page_html_to_bronze(
            company_url,
            worker_driver,
            bronze_dir=bronze_dir,
            max_retries=3,
            initial_delay=1.0,
            backoff_factor=2.0,
        )
        print(f"[WORKER DONE] {thread_name} completed {company_url}")
        return {"url": company_url, "saved_file": saved_path, "error": None, "thread_name": thread_name}
    except Exception as exc:
        print(f"[WORKER ERROR] {thread_name} failed {company_url}: {exc}")
        return {"url": company_url, "saved_file": None, "error": str(exc), "thread_name": thread_name}
    finally:
        worker_driver.quit()

In [26]:
def save_all_company_pages_to_bronze(
    company_profile_links,
    bronze_dir="../data/bronze",
):
    """Save all company profile HTML pages using parallel execution."""

    if not company_profile_links:
        raise ValueError("company_profile_links is empty. Run link collection first.")

    max_workers = HTML_SAVE_WORKERS
    print(f"Starting HTML save with {max_workers} workers...")

    saved_files = []
    failed_links = []
    total = len(company_profile_links)
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_map = {
            executor.submit(
                save_company_page_with_worker,
                company_url,
                bronze_dir,
            ): company_url
            for company_url in company_profile_links
        }

        for future in as_completed(future_map):
            completed += 1
            result = future.result()
            thread_name = result.get("thread_name", "unknown-thread")

            if result["error"] is None:
                saved_files.append(result["saved_file"])
                print(f"[PROGRESS] {completed}/{total} saved by {thread_name}")
            else:
                failed_links.append({"url": result["url"], "error": result["error"]})
                print(f"[PROGRESS] {completed}/{total} failed by {thread_name}: {result['url']}")

    summary = {
        "total_profiles": total,
        "saved_count": len(saved_files),
        "failed_count": len(failed_links),
        "saved_files": saved_files,
        "failed_links": failed_links,
    }

    print("HTML save finished.")
    print(f"Saved: {summary['saved_count']} | Failed: {summary['failed_count']}")

    return summary

In [ ]:
# Run pipeline
last_page_number = capture_number_pages(driver)
print(f"Last page number: {last_page_number}")

company_profile_links = collect_company_profile_links(url, last_page_number, driver)
print(f"Collected links: {len(company_profile_links)}")

save_summary = save_all_company_pages_to_bronze(
    company_profile_links,
    bronze_dir="../data/bronze",
)

print(f"Saved files: {save_summary['saved_count']}")
print(f"Failed files: {save_summary['failed_count']}")

Pagination detected on attempt 1. Last page: 203
Last page number: 203


KeyboardInterrupt: 